In [ ]:
!cp -r "/content/drive/MyDrive/Audio_data" "/content/audio"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
count = len([f for f in os.listdir('/content/audio') if f.endswith('.mp3')])
print(f"congratulations! Total {count} files downloaded.")

congratulations! Total 8588 files downloaded.


In [ ]:
!pip install --upgrade transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import os
import csv
import torch
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from tqdm import tqdm

# 1. Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Model & Feature Extractor
# We use FeatureExtractor instead of Processor to bypass the special_tokens error
model_name = 'kingabzpro/wav2vec2-large-xls-r-300m-Urdu'
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2Model.from_pretrained(model_name).to(device)

def extract_audio_features(audio_file):
    try:
        # Load audio (Wav2Vec2 requires exactly 16000Hz)
        y, sr = librosa.load(audio_file, sr=16000)

        # Truncate to 10 seconds to avoid CUDA Out-of-Memory (OOM) errors
        y = y[:16000*10]

        # Use feature_extractor (safe from the TypeError)
        inputs = feature_extractor(y, return_tensors="pt", sampling_rate=sr).to(device)

        with torch.no_grad():
            model_output = model(**inputs)

        # Mean pooling: converts sequence of vectors into one single vector (1024 dimensions)
        features = model_output.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        return features.tolist()
    except Exception as e:
        print(f"\nError processing {audio_file}: {e}")
        return None

def process_and_save(folder_path, output_file):
    # Get all mp3 files first so tqdm has a total count
    audio_tasks = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".mp3"):
                audio_tasks.append(os.path.join(root, file))

    print(f"Total audios found: {len(audio_tasks)}")

    with open(output_file, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        header_written = False

        # tqdm for progress tracking
        for audio_path in tqdm(audio_tasks, desc="Extracting Audio Features"):
            file_name = os.path.basename(audio_path).lower()

            # 3. Label Cleaning Logic (Matches your Video Logic)
            if 'anger' in file_name: label = 'Anger'
            elif 'sad' in file_name: label = 'Sad'
            elif 'love' in file_name: label = 'Love'
            elif 'happy' in file_name: label = 'Happy'
            elif 'neutral' in file_name or 'netural' in file_name: label = 'Neutral'
            else: label = 'Other'

            if label == 'Other': continue

            features = extract_audio_features(audio_path)

            if features:
                if not header_written:
                    # Header: Link, Label, then F_0 to F_1023
                    csv_writer.writerow(["Link", "Label"] + [f"F_{i}" for i in range(len(features))])
                    header_written = True

                csv_writer.writerow([audio_path, label] + features)

# --- EXECUTION ---
audio_directory = "/content/audio" # Ensure this path is correct
output_file = "/content/drive/MyDrive/audio_features.csv"

process_and_save(audio_directory, output_file)
print("\nProcessing Complete! File saved to Drive.")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: kingabzpro/wav2vec2-large-xls-r-300m-Urdu
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total audios found: 8588


Extracting Audio Features: 100%|██████████| 8588/8588 [17:31<00:00,  8.16it/s]


Processing Complete! File saved to Drive.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Check if CUDA (GPU) is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the data from your CSV file
data = pd.read_csv("/content/drive/MyDrive/audio_features.csv")
Y_AF = data["Label"]
# Split data into features (X) and labels (y)
X_AF = data.drop(columns=["Link", "Label"])


# Use LabelEncoder to convert string labels to numerical labels
label_encoder = LabelEncoder()
Y_AF = label_encoder.fit_transform(Y_AF)

sequence_length = 20
# --- Step 1: Split raw data first ---
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_AF.values, Y_AF, test_size=0.2, random_state=42, shuffle=False
)

def create_sequences(features, labels, seq_len):
    seqs, lbls = [], []
    for i in range(len(features) - seq_len + 1):
        seqs.append(features[i:i+seq_len])
        lbls.append(labels[i + seq_len - 1])
    return torch.tensor(seqs, dtype=torch.float32), torch.tensor(lbls, dtype=torch.long)

# --- Step 2: Create sequences separately ---
X_train_A, y_train_A = create_sequences(X_train_raw, y_train_raw, sequence_length)
X_test_A, y_test_A = create_sequences(X_test_raw, y_test_raw, sequence_length)

# Move to device
X_train_A, y_train_A = X_train_A.to(device), y_train_A.to(device)
X_test_A, y_test_A = X_test_A.to(device), y_test_A.to(device)
# Create a DataLoader for the training set (optional but useful for mini-batch training)
batch_size = 32  # Adjust as needed
train_dataset = TensorDataset(X_train_A, y_train_A)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Define an LSTM-based model
class EmotionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super(EmotionLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,dropout=dropout if num_layers > 1 else 0,bidirectional=True)
        self.fc = nn.Linear(hidden_size, num_classes)


        self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = torch.max(out, dim=1)[0]  # Take the output from the last time step
        out = self.fc(out)
        return out

# Define the LSTM model hyperparameters
input_size = X_train_A.shape[2]  # Input size based on the number of features in each time step
hidden_size = 64
num_layers = 2  # You can adjust this as needed
num_classes = len(label_encoder.classes_)

# Initialize the model and move it to the GPU
model = EmotionLSTM(input_size, hidden_size, num_layers, num_classes).to(device)

# Define a loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}')

# Set the model to evaluation mode
model.eval()

# Make predictions on the test set
with torch.no_grad():
    outputs = model(X_test_A)
    _, predicted = torch.max(outputs, 1)

# Move the predictions to the CPU and convert them to a NumPy array
predicted = predicted.cpu().numpy()

# Calculate accuracy
accuracy = accuracy_score(y_test_A.cpu().numpy(), predicted)
print("Accuracy:", accuracy)

Epoch [10/50], Loss: 1.3914
Epoch [20/50], Loss: 1.2596
Epoch [30/50], Loss: 1.2164
Epoch [40/50], Loss: 1.1525
Epoch [50/50], Loss: 1.1071
Accuracy: 0.5626839317245439
